## Menentukan Pertanyaan Bisnis

1. Pada jam berapa saja terjadi puncak penyewaan sepeda, dan bagaimana polanya berbeda antara hari kerja dan akhir pekan?
2. Bagaimana pengaruh kondisi cuaca dan musim terhadap jumlah penyewaan sepeda?

## Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
print('Library berhasil diimport.')

## Data Wrangling

### Gathering Data

Dataset: **Bike Sharing Dataset** dari Capital Bikeshare system, Washington D.C. (2011-2012).  
Sumber: [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/bike+sharing+dataset)

- `hour.csv`: data per jam (17.379 records)
- `day.csv`: data per hari (731 records)

In [ ]:
df_hour = pd.read_csv('data/hour.csv')
df_day  = pd.read_csv('data/day.csv')
print('=== hour.csv ===')
print(f'Shape : {df_hour.shape}')
print(df_hour.head())

In [ ]:
print('=== day.csv ===')
print(f'Shape : {df_day.shape}')
print(df_day.head())

### Assessing Data

Identifikasi permasalahan: missing value, duplikat, tipe data, dan outlier.

In [ ]:
df_hour.info()

In [ ]:
print('Missing value per kolom:')
print(df_hour.isnull().sum())
print(f'Duplikat: {df_hour.duplicated().sum()} baris')

In [ ]:
df_hour[['temp','atemp','hum','windspeed','casual','registered','cnt']].describe().round(2)

In [ ]:
Q1  = df_hour['cnt'].quantile(0.25)
Q3  = df_hour['cnt'].quantile(0.75)
IQR = Q3 - Q1
outliers = df_hour[(df_hour['cnt'] < Q1-1.5*IQR) | (df_hour['cnt'] > Q3+1.5*IQR)]
print(f'Outlier cnt: {len(outliers)} baris')
print('Outlier tidak dihapus - merupakan kondisi ekstrem valid (event kota).')

### Cleaning Data

Tidak ditemukan missing value maupun duplikat. Langkah cleaning:
1. Konversi `dteday` ke datetime
2. Tambah kolom label season, weather, weekday

In [ ]:
df_hour['dteday'] = pd.to_datetime(df_hour['dteday'])
df_day['dteday']  = pd.to_datetime(df_day['dteday'])

season_map  = {1:'Spring',2:'Summer',3:'Fall',4:'Winter'}
weather_map = {1:'Clear',2:'Mist/Cloudy',3:'Light Rain/Snow',4:'Heavy Rain'}
weekday_map = {0:'Senin',1:'Selasa',2:'Rabu',3:'Kamis',4:'Jumat',5:'Sabtu',6:'Minggu'}

df_hour['season_name']  = df_hour['season'].map(season_map)
df_hour['weather_name'] = df_hour['weathersit'].map(weather_map)
df_hour['weekday_name'] = df_hour['weekday'].map(weekday_map)
df_day['season_name']   = df_day['season'].map(season_map)
df_day['weather_name']  = df_day['weathersit'].map(weather_map)

print('Cleaning selesai.')
df_hour[['dteday','season_name','weather_name','weekday_name','cnt']].head()

## Exploratory Data Analysis (EDA)

### Pertanyaan 1: Pola Penyewaan per Jam

In [ ]:
hourly_workday = df_hour[df_hour['workingday']==1].groupby('hr')['cnt'].mean()
hourly_weekend = df_hour[df_hour['workingday']==0].groupby('hr')['cnt'].mean()

fig, ax = plt.subplots(figsize=(14,5))
ax.plot(hourly_workday.index, hourly_workday.values, 'o-', color='#2ecc71', lw=2.5, ms=6, label='Hari Kerja')
ax.plot(hourly_weekend.index, hourly_weekend.values, 's--', color='#9b59b6', lw=2.5, ms=6, label='Akhir Pekan')
ax.set_xlabel('Jam (0-23)'); ax.set_ylabel('Rata-rata Penyewaan')
ax.set_title('Pola Penyewaan Sepeda per Jam: Hari Kerja vs Akhir Pekan', fontsize=14, fontweight='bold')
ax.set_xticks(range(24)); ax.legend(fontsize=11); ax.grid(alpha=0.4)
plt.tight_layout(); plt.savefig('pola_jam.png', dpi=150, bbox_inches='tight'); plt.show()
print(f'Jam puncak hari kerja : jam {hourly_workday.idxmax()} ({hourly_workday.max():.0f}/jam)')
print(f'Jam puncak akhir pekan: jam {hourly_weekend.idxmax()} ({hourly_weekend.max():.0f}/jam)')

### Pertanyaan 2: Pengaruh Cuaca & Musim

In [ ]:
season_order  = ['Spring','Summer','Fall','Winter']
season_colors = ['#3498db','#f39c12','#e74c3c','#9b59b6']

fig, axes = plt.subplots(1,2, figsize=(14,5))
season_data = [df_hour[df_hour['season_name']==s]['cnt'].values for s in season_order]
bp = axes[0].boxplot(season_data, patch_artist=True, medianprops=dict(color='white',linewidth=2))
for patch, color in zip(bp['boxes'], season_colors): patch.set_facecolor(color); patch.set_alpha(0.8)
axes[0].set_xticklabels(season_order)
axes[0].set_title('Distribusi Penyewaan per Musim', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Jumlah Penyewaan (cnt)'); axes[0].grid(axis='y', alpha=0.4)

season_mean = df_hour.groupby('season_name')['cnt'].mean().reindex(season_order)
bars = axes[1].bar(season_order, season_mean.values, color=season_colors, alpha=0.85, width=0.6)
axes[1].set_title('Rata-rata Penyewaan per Musim', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Rata-rata cnt'); axes[1].grid(axis='y', alpha=0.4)
for bar, val in zip(bars, season_mean.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, val+2, f'{val:.0f}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.savefig('distribusi_musim.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
weather_order  = ['Clear','Mist/Cloudy','Light Rain/Snow','Heavy Rain']
weather_colors = ['#27ae60','#f39c12','#3498db','#e74c3c']
weather_mean   = df_hour.groupby('weather_name')['cnt'].mean()
avail = [w for w in weather_order if w in weather_mean.index]
weather_mean = weather_mean.reindex(avail)

fig, ax = plt.subplots(figsize=(10,5))
bars = ax.bar(range(len(weather_mean)), weather_mean.values, color=weather_colors[:len(weather_mean)], alpha=0.85, width=0.55)
ax.set_xticks(range(len(weather_mean))); ax.set_xticklabels(weather_mean.index, rotation=10, ha='right')
ax.set_title('Rata-rata Penyewaan per Kondisi Cuaca', fontsize=14, fontweight='bold')
ax.set_ylabel('Rata-rata cnt'); ax.grid(axis='y', alpha=0.4)
for bar, val in zip(bars, weather_mean.values):
    ax.text(bar.get_x()+bar.get_width()/2, val+2, f'{val:.0f}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.savefig('pengaruh_cuaca.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10,7))
corr = df_hour[['temp','atemp','hum','windspeed','casual','registered','cnt']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=ax, square=True, linewidths=0.5)
ax.set_title('Heatmap Korelasi Antar Variabel Numerik', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('heatmap_korelasi.png', dpi=150, bbox_inches='tight'); plt.show()

## Visualization & Explanatory Analysis

### Visualisasi 1: Tren Harian 2011-2012

In [ ]:
fig, ax = plt.subplots(figsize=(14,5))
ax.plot(df_day['dteday'], df_day['cnt'], color='#3498db', lw=1.2, alpha=0.7)
ax.plot(df_day['dteday'], df_day['cnt'].rolling(30).mean(), color='#e74c3c', lw=2.5, label='MA-30 hari')
ax.set_title('Tren Penyewaan Sepeda Harian (2011-2012)', fontsize=14, fontweight='bold')
ax.set_xlabel('Tanggal'); ax.set_ylabel('Total Penyewaan per Hari')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('tren_harian.png', dpi=150, bbox_inches='tight'); plt.show()

### Visualisasi 2: Heatmap Jam x Hari

In [ ]:
pivot = df_hour.groupby(['hr','weekday'])['cnt'].mean().unstack()
pivot.columns = ['Sen','Sel','Rab','Kam','Jum','Sab','Min']
fig, ax = plt.subplots(figsize=(13,6))
sns.heatmap(pivot.T, cmap='YlOrRd', linewidths=0.3, ax=ax, cbar_kws={'label':'Rata-rata Penyewaan','shrink':0.85})
ax.set_title('Heatmap Rata-rata Penyewaan: Jam x Hari', fontsize=14, fontweight='bold')
ax.set_xlabel('Jam (0-23)'); ax.set_ylabel('Hari')
plt.tight_layout(); plt.savefig('heatmap_jam_hari.png', dpi=150, bbox_inches='tight'); plt.show()

## Analisis Lanjutan: Clustering dengan Manual Grouping

Tiga dimensi clustering tanpa algoritma ML:
1. **Clustering Intensitas Jam** — 5 tier demand
2. **Clustering Segmentasi Hari** — workday peak / normal / weekend
3. **Clustering Kondisi Operasional** — musim + cuaca

### Clustering 1: Intensitas Jam

| Cluster | Threshold | Jam | Makna Bisnis |
|---------|-----------|-----|--------------|
| Very Low | cnt < 50 | 01-05 | Dini hari — operasional minimum |
| Low | 50-100 | 00, 06, 23 | Transisi pagi/malam |
| Medium | 100-220 | 07, 09-11, 21-22 | Penggunaan moderat |
| High | 220-350 | 12-16, 19-20 | Siang & awal malam ramai |
| Very High | > 350 | 08, 17-18 | Jam puncak komuter |

In [ ]:
hourly_avg = df_hour.groupby('hr')['cnt'].mean()

def cluster_jam(cnt):
    if cnt < 50:    return 'Very Low'
    elif cnt < 100: return 'Low'
    elif cnt < 220: return 'Medium'
    elif cnt < 350: return 'High'
    else:           return 'Very High'

CLUSTER_ORDER = ['Very Low','Low','Medium','High','Very High']
COLOR = {
    'Very Low':'#3a4a6b', 'Low':'#4e7fba', 'Medium':'#f0a500',
    'High':'#e05c3a', 'Very High':'#c0392b',
    'Workday Peak':'#e74c3c', 'Workday Normal':'#27ae60', 'Weekend':'#9b59b6',
    'Kondisi Ideal':'#27ae60','Kondisi Sedang':'#f39c12','Kondisi Buruk':'#e74c3c',
}

hourly_cluster = hourly_avg.apply(cluster_jam).reset_index()
hourly_cluster.columns = ['hr','intensity_cluster']
df_hour = df_hour.merge(hourly_cluster, on='hr')
df_hour['intensity_cluster'] = pd.Categorical(df_hour['intensity_cluster'], categories=CLUSTER_ORDER, ordered=True)

print('Hasil Clustering Intensitas Jam:')
print('-'*65)
for c in CLUSTER_ORDER:
    hrs = hourly_cluster[hourly_cluster['intensity_cluster']==c]['hr'].tolist()
    avg = df_hour[df_hour['intensity_cluster']==c]['cnt'].mean()
    n   = (df_hour['intensity_cluster']==c).sum()
    print(f'  {c:10s} | Jam {str(hrs):35s} | N={n:5,} | Avg={avg:.1f}')

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(16,5))
hrs    = hourly_cluster['hr'].values
avgs   = [hourly_avg[h] for h in hrs]
colors = [COLOR[c] for c in hourly_cluster['intensity_cluster']]
axes[0].bar(hrs, avgs, color=colors, width=0.75, alpha=0.88)
for thresh, lbl in [(50,'50'),(100,'100'),(220,'220'),(350,'350')]:
    axes[0].axhline(thresh, color='gray', ls='--', lw=0.9, alpha=0.5)
    axes[0].text(23.7, thresh+4, lbl, fontsize=8, color='gray')
axes[0].set_title('Rata-rata Penyewaan per Jam\n(warna = cluster intensitas)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Jam (0-23)'); axes[0].set_ylabel('Rata-rata cnt')
axes[0].set_xticks(range(0,24,2))
patches = [mpatches.Patch(color=COLOR[c], label=c) for c in CLUSTER_ORDER]
axes[0].legend(handles=patches, fontsize=9, loc='upper left')

data_box = [df_hour[df_hour['intensity_cluster']==c]['cnt'].values for c in CLUSTER_ORDER]
bp = axes[1].boxplot(data_box, patch_artist=True, medianprops=dict(color='black',linewidth=2))
for patch, c in zip(bp['boxes'], CLUSTER_ORDER): patch.set_facecolor(COLOR[c]); patch.set_alpha(0.82)
axes[1].set_xticklabels(CLUSTER_ORDER, rotation=20, ha='right', fontsize=9)
axes[1].set_title('Distribusi cnt per Cluster Intensitas', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Jumlah Penyewaan (cnt)'); axes[1].grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.savefig('clustering_intensitas_jam.png', dpi=150, bbox_inches='tight'); plt.show()

**Insight Clustering 1:**
- **Very High** (jam 8, 17, 18): Jam puncak komuter ~415/jam — waktu kritis untuk ketersediaan sepeda.
- **Very Low** (jam 1-5): Hanya ~19/jam — waktu ideal untuk maintenance dan redistribusi armada.
- Pola bimodal pagi-sore khas **commuter city**.

### Clustering 2: Segmentasi Hari

In [ ]:
def cluster_hari(row):
    if row['weekday'] >= 5:               return 'Weekend'
    elif row['hr'] in [7,8,9,16,17,18,19]: return 'Workday Peak'
    else:                                  return 'Workday Normal'

df_hour['day_segment'] = df_hour.apply(cluster_hari, axis=1)

print('Hasil Clustering Segmentasi Hari:')
print('-'*58)
for seg in ['Workday Peak','Weekend','Workday Normal']:
    sub = df_hour[df_hour['day_segment']==seg]
    print(f'  {seg:16s} | N={len(sub):5,} | Avg={sub["cnt"].mean():.1f} | Max={sub["cnt"].max()}')

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14,5))
seg_order = ['Workday Normal','Weekend','Workday Peak']
seg_mean  = df_hour.groupby('day_segment')['cnt'].mean().reindex(seg_order)
bars = axes[0].barh(seg_order, seg_mean.values, color=[COLOR[s] for s in seg_order], height=0.45, alpha=0.88)
axes[0].set_xlabel('Rata-rata Penyewaan')
axes[0].set_title('Rata-rata Penyewaan per Segmen Hari', fontsize=13, fontweight='bold')
axes[0].grid(axis='x', alpha=0.4)
for bar, val in zip(bars, seg_mean.values):
    axes[0].text(val+3, bar.get_y()+bar.get_height()/2, f'{val:.0f}', va='center', fontsize=12, fontweight='bold')

for seg, color, ls in zip(seg_order, [COLOR[s] for s in seg_order], ['-','--','-.']):
    hmean = df_hour[df_hour['day_segment']==seg].groupby('hr')['cnt'].mean()
    axes[1].plot(hmean.index, hmean.values, color=color, lw=2.2, linestyle=ls, label=seg, marker='o', ms=4)
axes[1].set_title('Profil Penyewaan per Jam per Segmen', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Jam (0-23)'); axes[1].set_ylabel('Rata-rata cnt')
axes[1].set_xticks(range(0,24,2)); axes[1].grid(alpha=0.4); axes[1].legend(fontsize=10)
plt.tight_layout(); plt.savefig('clustering_segmentasi_hari.png', dpi=150, bbox_inches='tight'); plt.show()

**Insight Clustering 2:**
- **Workday Peak** rata-rata **343/jam** — hampir 3x lipat Workday Normal.
- **Weekend** (193/jam) lebih tinggi dari Workday Normal — puncak di siang hari (leisure).
- Profil per jam: hari kerja bimodal (pagi & sore), akhir pekan unimodal (siang).

### Clustering 3: Kondisi Operasional (Cuaca + Musim)

In [ ]:
def cluster_kondisi(row):
    if row['season'] in [2,3] and row['weathersit'] in [1,2]: return 'Kondisi Ideal'
    elif row['weathersit'] >= 3 or row['season'] == 1:         return 'Kondisi Buruk'
    else:                                                        return 'Kondisi Sedang'

df_hour['condition_cluster'] = df_hour.apply(cluster_kondisi, axis=1)

print('Hasil Clustering Kondisi Operasional:')
print('-'*55)
ideal_avg = df_hour[df_hour['condition_cluster']=='Kondisi Ideal']['cnt'].mean()
for cond in ['Kondisi Ideal','Kondisi Sedang','Kondisi Buruk']:
    sub = df_hour[df_hour['condition_cluster']==cond]
    avg = sub['cnt'].mean()
    pct = (avg/ideal_avg - 1)*100
    print(f'  {cond:18s} | N={len(sub):5,} | Avg={avg:.1f} | vs Ideal: {pct:+.1f}%')

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14,5))
cond_order = ['Kondisi Ideal','Kondisi Sedang','Kondisi Buruk']
cond_mean  = df_hour.groupby('condition_cluster')['cnt'].mean().reindex(cond_order)
bars = axes[0].bar(['Ideal','Sedang','Buruk'], cond_mean.values, color=[COLOR[c] for c in cond_order], width=0.5, alpha=0.88)
axes[0].set_ylabel('Rata-rata Penyewaan'); axes[0].grid(axis='y', alpha=0.4)
axes[0].set_title('Rata-rata Penyewaan per Kluster Kondisi', fontsize=13, fontweight='bold')
for bar, val in zip(bars, cond_mean.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, val+2, f'{val:.0f}', ha='center', fontsize=12, fontweight='bold')

cross = pd.crosstab(df_hour['season_name'], df_hour['condition_cluster'], normalize='index')*100
cross = cross.reindex(['Spring','Summer','Fall','Winter'])
cond_c = {'Kondisi Ideal':'#27ae60','Kondisi Sedang':'#f39c12','Kondisi Buruk':'#e74c3c'}
bottom = np.zeros(4)
for cond in cond_order:
    if cond in cross.columns:
        axes[1].bar(cross.index, cross[cond].values, bottom=bottom, color=cond_c[cond], label=cond.replace('Kondisi ',''), alpha=0.88)
        bottom += cross[cond].values
axes[1].set_title('Proporsi Cluster Kondisi per Musim (%)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Persentase (%)'); axes[1].legend(fontsize=10)
plt.tight_layout(); plt.savefig('clustering_kondisi.png', dpi=150, bbox_inches='tight'); plt.show()

**Insight Clustering 3:**
- **Kondisi Buruk** menyebabkan penurunan **~50%** dibanding Kondisi Ideal.
- Musim panas dan gugur adalah **golden season** — mayoritas Kondisi Ideal.
- Musim semi (Spring) paling tidak stabil — didominasi Kondisi Buruk.

## Conclusion

### Pertanyaan 1: Pola Penyewaan per Jam

Terdapat dua jam puncak pada hari kerja: **jam 08.00** (~359/jam) dan **jam 17.00-18.00** (~425-462/jam). Pola ini khas commuter city — mayoritas pengguna terdaftar menggunakan sepeda untuk komuter. Pada akhir pekan, puncak terjadi di **jam 12.00-15.00** (~300/jam) mencerminkan penggunaan leisure.

### Pertanyaan 2: Pengaruh Cuaca & Musim

Kondisi cuaca dan musim berdampak signifikan:
- **Kondisi Ideal** (panas/gugur + cerah): ~229/jam
- **Kondisi Buruk** (hujan/salju atau musim dingin): ~115/jam — **penurunan ~50%**
- Musim gugur (Fall) adalah musim terbaik; musim semi (Spring) terendah

### Rekomendasi Bisnis

1. **Distribusi Armada**: Siapkan stok sebelum jam Very High (07.00 dan 16.00)
2. **Harga Dinamis**: Tarif premium saat Workday Peak, diskon saat Very Low (malam)
3. **Jadwal Maintenance**: Lakukan di jam Very Low (01.00-05.00)
4. **Strategi Cuaca**: Kurangi stok ~50% saat Kondisi Buruk untuk efisiensi
5. **Program Weekend**: Paket promosi leisure untuk meningkatkan utilisasi siang hari